# XGBoost Price Prediction Model - Second Life

Dự đoán giá sản phẩm đồ cũ dựa trên dataset thu thập từ Chợ Tốt — **23 danh mục** trải dài từ điện tử, thời trang, nội thất, xe cộ đến mỹ phẩm.

## Đầu vào của mô hình

| Feature | Mô tả | Bắt buộc |
|---------|-------|----------|
| `sl_category_name` | Nhóm danh mục lớn | ✅ |
| `sl_sub_category_name` | Danh mục con | ✅ |
| `brand` | Thương hiệu | ✅ |
| `model` | Tên model / sản phẩm | — |
| `sl_condition` | Tình trạng: `New` / `Like New` / `Good` | — |
| `origin_label` | Nguồn gốc: `Chính hãng`, `Nhập khẩu`, `Không rõ`… | — |
| `warranty_label` | Bảo hành: `Còn bảo hành`, `Hết bảo hành`, `Không bảo hành` | — |
| `manufacture_year` | Năm sản xuất | — |
| `region_name` | Tỉnh/Thành phố | — |
| `capacity_gb` | Dung lượng GB *(chỉ dùng cho điện tử)* | — |

## 23 Sub-categories được hỗ trợ

| Nhóm | Sub-category |
|------|-------------|
| **Điện tử - Điện máy** | Điện thoại & Phụ kiện, Máy tính bảng, Máy tính & Laptop, Tivi & Màn hình, Máy ảnh & Quay phim, Thiết bị gia dụng |
| **Thời trang & Phụ kiện** | Quần áo nữ, Quần áo nam, Giày dép, Túi xách & Balo |
| **Nhà cửa & Nội thất** | Nội thất phòng khách, Đồ dùng nhà bếp, Nội thất phòng ngủ |
| **Xe cộ & Phương tiện** | Xe máy, Xe đạp & Xe điện, Phụ tùng xe |
| **Thể thao & Du lịch** | Dụng cụ thể thao & Gym |
| **Sách & Giải trí** | Game & Console, Sách cũ |
| **Mẹ & Bé** | Đồ dùng cho bé |
| **Mỹ phẩm & Chăm sóc** | Mỹ phẩm & Skincare |

> **Output:** `price_vnd` — giá dự đoán (VND)

### Google Colab

1. Upload `pricing_dataset.csv` to Drive: `MyDrive/SecondLife/pricing_dataset.csv` **or** upload in cell 0.
2. **Runtime → Run all** — model saves to `MyDrive/SecondLife/pricing_model/` and downloads `pricing_model_artifacts.zip`.
3. Copy the `model/` folder into repo: `scripts/pricing/model/` for backend integration.

## 0. Google Colab setup

In [ ]:
import os
import sys
import shutil
from datetime import datetime, timezone
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

# --- Edit paths on Colab if needed ---
USE_GOOGLE_DRIVE = True
DRIVE_DATA_PATH = '/content/drive/MyDrive/SecondLife/pricing_dataset.csv'
DRIVE_MODEL_DIR = '/content/drive/MyDrive/SecondLife/pricing_model'

WORK_DIR = Path('/content/second_life_pricing') if IN_COLAB else Path('.')
DATA_PATH = WORK_DIR / 'data' / 'pricing_dataset.csv'
MODEL_DIR = WORK_DIR / 'model'

if IN_COLAB:
  get_ipython().system('pip install -q xgboost scikit-learn joblib seaborn matplotlib')

WORK_DIR.mkdir(parents=True, exist_ok=True)
(WORK_DIR / 'data').mkdir(exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

if IN_COLAB:
  os.chdir(WORK_DIR)
  print('Working directory:', WORK_DIR.resolve())

  if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    drive_data = Path(DRIVE_DATA_PATH)
    if drive_data.exists():
      shutil.copy2(drive_data, DATA_PATH)
      print('Dataset copied from Drive ->', DATA_PATH)
    else:
      print('Drive file not found:', drive_data)
      print('Upload pricing_dataset.csv (browser dialog)...')
      from google.colab import files
      uploaded = files.upload()
      csv_name = next((n for n in uploaded if n.endswith('.csv')), None)
      if not csv_name:
        raise FileNotFoundError('Please upload pricing_dataset.csv')
      DATA_PATH.write_bytes(uploaded[csv_name])
      print('Saved upload ->', DATA_PATH)
else:
  # Local: scripts/pricing/data/pricing_dataset.csv
  if not DATA_PATH.exists():
    DATA_PATH = Path('data/pricing_dataset.csv')
  print('Working directory:', Path.cwd())
  print('Dataset path:', DATA_PATH.resolve())

assert DATA_PATH.exists(), f'Missing dataset: {DATA_PATH}'
print('Ready. IN_COLAB =', IN_COLAB)


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline

import xgboost as xgb
import joblib
import json

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
print('XGBoost version:', xgb.__version__)


## 2. Load & Explore Data

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'Dataset: {DATA_PATH}')
print(f'Dataset shape: {df.shape}')
df.head(3)


In [ ]:
print('=== Thông tin dataset ===')
print(f'Tổng số bản ghi: {len(df)}')
print(f'\n--- Phân bố theo nhóm danh mục ---')
print(df['sl_category_name'].value_counts())
print(f'\n--- Phân bố theo sub-category ---')
print(df['sl_sub_category_name'].value_counts())
print(f'\n--- Thương hiệu top 10 ---')
print(df['brand'].value_counts().head(10))
print(f'\n--- Tình trạng ---')
print(df['sl_condition'].value_counts())
print(f'\n--- Phân bố giá ---')
print(df['price_vnd'].describe().apply(lambda x: f'{x:,.0f}'))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Phân bố giá gốc
axes[0].hist(df['price_vnd'] / 1e6, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_xlabel('Giá (triệu VND)')
axes[0].set_ylabel('Số lượng')
axes[0].set_title('Phân bố giá (gốc)')

# Phân bố giá log
axes[1].hist(np.log1p(df['price_vnd']), bins=50, color='coral', edgecolor='white', alpha=0.8)
axes[1].set_xlabel('log(giá + 1)')
axes[1].set_ylabel('Số lượng')
axes[1].set_title('Phân bố giá (log scale)')

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Giá trung vị theo sub-category (horizontal bar — dễ đọc với 23 danh mục)
sub_price = df.groupby('sl_sub_category_name')['price_vnd'].median().sort_values() / 1e6
colors = plt.cm.tab20.colors
sub_price.plot(kind='barh', ax=axes[0], color=colors[:len(sub_price)], edgecolor='white')
axes[0].set_xlabel('Giá trung vị (triệu VND)')
axes[0].set_title('Giá trung vị theo Sub-category')
axes[0].axvline(sub_price.mean(), color='red', linestyle='--', alpha=0.6, label=f'TB = {sub_price.mean():.1f}M')
axes[0].legend()

# Số lượng mẫu theo nhóm danh mục lớn
grp_count = df.groupby('sl_category_name').size().sort_values(ascending=False)
grp_count.plot(kind='bar', ax=axes[1], color='coral', edgecolor='white', alpha=0.85)
axes[1].set_xlabel('Nhóm danh mục')
axes[1].set_ylabel('Số lượng mẫu')
axes[1].set_title('Số lượng mẫu theo nhóm danh mục')
axes[1].tick_params(axis='x', rotation=30)
for i, v in enumerate(grp_count):
    axes[1].text(i, v + 5, str(v), ha='center', fontsize=10)

plt.tight_layout()
plt.show()


## 3. Feature Engineering

In [ ]:
# --- Training scope (MVP): exclude categories that destroy global MAPE (esp. Ô tô after remap).
TRAIN_SUB_CATEGORIES = {
    'Điện thoại & Phụ kiện', 'Máy tính bảng', 'Máy tính & Laptop',
    'Tivi & Màn hình', 'Máy ảnh & Quay phim', 'Thiết bị gia dụng',
    'Dụng cụ thể thao & Gym', 'Nội thất phòng ngủ',
    'Xe đạp & Xe điện', 'Xe máy',
}
EXCLUDE_FROM_TRAIN = {
    'Ô tô', 'Túi xách & Balo', 'Quần áo nam', 'Quần áo nữ', 'Giày dép',
    'Sách cũ', 'Đồ dùng nhà bếp', 'Nội thất phòng khách', 'Game & Console',
    'Mỹ phẩm & Skincare', 'Đồ dùng cho bé', 'Phụ tùng xe',
}
MAPE_WHITELIST_MAX = 75.0

ELECTRONICS_CATS = {
    'Điện thoại & Phụ kiện', 'Máy tính bảng', 'Máy tính & Laptop',
    'Tivi & Màn hình', 'Máy ảnh & Quay phim', 'Thiết bị gia dụng',
}

CURRENT_YEAR = 2026

print(f'Full dataset: {len(df)} rows')
df_train_pool = df[~df['sl_sub_category_name'].isin(EXCLUDE_FROM_TRAIN)].copy()
if TRAIN_SUB_CATEGORIES:
    df_train_pool = df_train_pool[df_train_pool['sl_sub_category_name'].isin(TRAIN_SUB_CATEGORIES)]
print(f'Training pool: {len(df_train_pool)} rows')
print(df_train_pool['sl_sub_category_name'].value_counts())


def preprocess(df: pd.DataFrame, is_train: bool = True) -> pd.DataFrame:
    d = df.copy()

    # --- Lọc outlier giá theo từng nhóm danh mục (chỉ khi train) ---
    if is_train:
        before = len(d)
        clean_parts = []
        for cat, grp in d.groupby('sl_category_name'):
            q_low  = grp['price_vnd'].quantile(0.01)
            q_high = grp['price_vnd'].quantile(0.99)
            clean_parts.append(grp[(grp['price_vnd'] >= q_low) & (grp['price_vnd'] <= q_high)])
        d = pd.concat(clean_parts).sort_index()
        print(f'Sau lọc outlier theo danh mục: {len(d)} / {before} bản ghi')

    # --- Dung lượng → số (GB); 0 cho danh mục không có capacity ---
    d['capacity_gb'] = (
        d['capacity'].fillna('').astype(str)
        .str.extract(r'(\d+)')[0]
        .astype(float)
        .fillna(0)
    )

    # --- Cờ "có phải điện tử?" ---
    d['is_electronics'] = d['sl_sub_category_name'].isin(ELECTRONICS_CATS).astype(int)

    # --- capacity_gb chỉ giữ giá trị thực cho điện tử, còn lại = 0 ---
    d.loc[d['is_electronics'] == 0, 'capacity_gb'] = 0

    # --- Năm sản xuất & tuổi sản phẩm ---
    d['manufacture_year'] = pd.to_numeric(d['manufacture_year'], errors='coerce').fillna(0).astype(int)
    d['product_age'] = d['manufacture_year'].apply(lambda y: CURRENT_YEAR - y if y > 2000 else 5)

    # --- Fill missing cho categorical ---
    cat_fill_map = {
        'sl_category_name':     'Unknown',
        'sl_sub_category_name': 'Unknown',
        'brand':                'OTHER',
        'model':                'Unknown',
        'sl_condition':         'Good',
        'origin_label':         'Không rõ',
        'warranty_label':       'Không bảo hành',
        'region_name':          'Unknown',
    }
    for col, fill_val in cat_fill_map.items():
        d[col] = d[col].fillna(fill_val).replace('', fill_val)

    return d


FEATURE_COLS = [
    'sl_category_name',      # nhóm danh mục lớn (Electronics, Fashion, Home…)
    'sl_sub_category_name',  # danh mục con
    'brand',                 # thương hiệu
    'model',                 # model / tên sản phẩm
    'capacity_gb',           # dung lượng GB (điện tử); 0 cho danh mục khác
    'is_electronics',        # cờ nhóm điện tử
    'sl_condition',          # tình trạng
    'origin_label',          # nguồn gốc
    'warranty_label',        # bảo hành
    'manufacture_year',      # năm sản xuất
    'product_age',           # tuổi sản phẩm
    'region_name',           # tỉnh/thành
]

CAT_COLS = [
    'sl_category_name', 'sl_sub_category_name',
    'brand', 'model',
    'sl_condition', 'origin_label', 'warranty_label', 'region_name',
]

TARGET = 'price_vnd'

df_clean = preprocess(df_train_pool, is_train=True)
df_all_clean = preprocess(df, is_train=False)
print(f'Features: {len(FEATURE_COLS)} | Train: {len(df_clean)} | Full (eval only): {len(df_all_clean)}')
print(df_clean[FEATURE_COLS].dtypes)


## 4. Encode Features

In [ ]:
X = df_clean[FEATURE_COLS].copy()
y = np.log1p(df_clean[TARGET])  # log-transform giá để cải thiện phân bố

# Label encoding cho các cột categorical
label_encoders = {}
for col in CAT_COLS:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    label_encoders[col] = le

print('Feature matrix shape:', X.shape)
X.head(3)


## 5. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=df_clean['sl_sub_category_name'],
)
print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]} (stratified)')


## 6. Train XGBoost Model

In [ ]:
params = {
    'n_estimators':     500,
    'max_depth':        6,
    'learning_rate':    0.05,
    'subsample':        0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 3,
    'reg_alpha':        0.1,
    'reg_lambda':       1.0,
    'random_state':     42,
    'n_jobs':           -1,
    'early_stopping_rounds': 30,
}

model = xgb.XGBRegressor(**params)
model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50
)
print('\nBest iteration:', model.best_iteration)


## 7. Evaluation

In [ ]:
y_pred_log = model.predict(X_test)
y_pred     = np.expm1(y_pred_log)   # inverse log
y_true     = np.expm1(y_test)

mae   = mean_absolute_error(y_true, y_pred)
rmse  = np.sqrt(mean_squared_error(y_true, y_pred))
r2    = r2_score(y_true, y_pred)
mape  = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

print('=== Model Performance (Test Set) ===')
print(f'  MAE  : {mae:>15,.0f} VND  (~{mae/1e6:.2f} triệu)')
print(f'  RMSE : {rmse:>15,.0f} VND  (~{rmse/1e6:.2f} triệu)')
print(f'  MAPE : {mape:>14.1f} %')
print(f'  R²   : {r2:>14.4f}')


In [ ]:
# Cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_model = xgb.XGBRegressor(
    n_estimators=model.best_iteration + 1,
    max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    min_child_weight=3, random_state=42, n_jobs=-1
)
cv_scores = cross_val_score(cv_model, X, y, cv=kf, scoring='r2')
print(f'5-Fold CV R²: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'Scores: {cv_scores.round(4)}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Predicted vs Actual
axes[0].scatter(y_true / 1e6, y_pred / 1e6, alpha=0.4, s=20, color='steelblue')
max_val = max(y_true.max(), y_pred.max()) / 1e6
axes[0].plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='Perfect fit')
axes[0].set_xlabel('Giá thực tế (triệu VND)')
axes[0].set_ylabel('Giá dự đoán (triệu VND)')
axes[0].set_title(f'Predicted vs Actual (R²={r2:.3f})')
axes[0].legend()

# Residuals
residuals = (y_pred - y_true) / 1e6
axes[1].hist(residuals, bins=50, color='coral', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Residual (triệu VND)')
axes[1].set_ylabel('Số lượng')
axes[1].set_title(f'Phân bố sai số (MAE={mae/1e6:.2f}M, MAPE={mape:.1f}%)')

plt.tight_layout()
plt.show()


## 8. Feature Importance

In [ ]:
feat_imp = pd.Series(model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
feat_imp.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_xlabel('Feature Importance (gain)')
ax.set_title('XGBoost Feature Importance')
plt.tight_layout()
plt.show()

print('\nFeature importance ranking:')
for name, imp in feat_imp.sort_values(ascending=False).items():
    print(f'  {name:<25} {imp:.4f}')


## 9. Per-Category Performance

In [ ]:
# Use .loc (label index), not .iloc — rows keep original CSV indices after outlier filter
test_df = df_clean.loc[X_test.index].copy()
test_df['price_predicted'] = y_pred
test_df['abs_error'] = np.abs(test_df['price_vnd'] - test_df['price_predicted'])
test_df['error_pct'] = test_df['abs_error'] / test_df['price_vnd'] * 100

def perf_agg(grp):
    return pd.Series({
        'count': len(grp),
        'median_price_M': grp['price_vnd'].median() / 1e6,
        'mape_%': grp['error_pct'].mean(),
        'mae_M': grp['abs_error'].mean() / 1e6,
    })

# Hiệu suất theo nhóm lớn
print('=== Hiệu suất theo nhóm danh mục ===')
grp_perf = test_df.groupby('sl_category_name').apply(perf_agg).round(2).sort_values('mape_%')
print(grp_perf.to_string())

# Hiệu suất theo sub-category
print('\n=== Hiệu suất theo sub-category ===')
sub_perf = test_df.groupby('sl_sub_category_name').apply(perf_agg).round(2).sort_values('mape_%')
print(sub_perf.to_string())

price_suggestion_subcategories = sub_perf[sub_perf['mape_%'] <= MAPE_WHITELIST_MAX].index.tolist()
print(f'\n=== Sub-categories OK for price suggestion (MAPE <= {MAPE_WHITELIST_MAX}%) ===')
print(price_suggestion_subcategories)
print(f'\nExcluded from training: {sorted(EXCLUDE_FROM_TRAIN)}')

# Visualize MAPE theo sub-category
fig, ax = plt.subplots(figsize=(12, 8))
sub_perf_plot = sub_perf.sort_values('mape_%')
colors_bar = ['#2ecc71' if v < 20 else '#f39c12' if v < 40 else '#e74c3c' for v in sub_perf_plot['mape_%']]
sub_perf_plot['mape_%'].plot(kind='barh', ax=ax, color=colors_bar, edgecolor='white')
ax.axvline(20, color='green', linestyle='--', alpha=0.5, label='20% (tốt)')
ax.axvline(40, color='orange', linestyle='--', alpha=0.5, label='40% (trung bình)')
ax.set_xlabel('MAPE (%)')
ax.set_title('MAPE theo Sub-category (xanh<20%, cam 20-40%, đỏ>40%)')
ax.legend()
plt.tight_layout()
plt.show()


## 10. Save Model & Encoders

In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)
model_path = MODEL_DIR / 'xgboost_price_model.json'
encoders_path = MODEL_DIR / 'label_encoders.pkl'
metadata_path = MODEL_DIR / 'model_metadata.json'

model.save_model(model_path)
print('Model saved ->', model_path)

joblib.dump(label_encoders, encoders_path)
print('Encoders saved ->', encoders_path)

metadata = {
    'feature_cols': FEATURE_COLS,
    'cat_cols': CAT_COLS,
    'target': TARGET,
    'trained_at': datetime.now(timezone.utc).isoformat(),
    'dataset_path': str(DATA_PATH),
    'train_rows': int(len(df_clean)),
    'metrics': {
        'mae': round(mae, 0),
        'rmse': round(rmse, 0),
        'r2': round(r2, 4),
        'mape_pct': round(mape, 2),
    },
    'categories': {
        col: list(label_encoders[col].classes_)
        for col in CAT_COLS
    },
    'electronics_cats': list(ELECTRONICS_CATS),
    'current_year': CURRENT_YEAR,
    'train_sub_categories': sorted(TRAIN_SUB_CATEGORIES),
    'exclude_from_train': sorted(EXCLUDE_FROM_TRAIN),
    'price_suggestion_subcategories': price_suggestion_subcategories,
    'subcategory_test_mape': sub_perf['mape_%'].to_dict(),
}
with metadata_path.open('w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)
print('Metadata saved ->', metadata_path)

# --- Colab: persist to Google Drive + download zip ---
if IN_COLAB:
    if USE_GOOGLE_DRIVE:
        drive_model_dir = Path(DRIVE_MODEL_DIR)
        drive_model_dir.mkdir(parents=True, exist_ok=True)
        for src in (model_path, encoders_path, metadata_path):
            shutil.copy2(src, drive_model_dir / src.name)
        print('Copied to Google Drive ->', drive_model_dir)

    zip_base = WORK_DIR / 'pricing_model_artifacts'
    shutil.make_archive(str(zip_base), 'zip', MODEL_DIR)
    zip_file = Path(str(zip_base) + '.zip')
    print('Zip created ->', zip_file)

    from google.colab import files
    files.download(str(zip_file))
    print('Download started: pricing_model_artifacts.zip')
else:
    print('Local artifacts in ->', MODEL_DIR.resolve())
    print('Copy to repo: scripts/pricing/model/')


## 11. Prediction Function (Inference)

In [ ]:
def predict_price(input_data: dict) -> dict:
    """
    Dự đoán giá sản phẩm — hỗ trợ toàn bộ 23 danh mục.

    Parameters (dict)
    -----------------
    Bắt buộc:
        sl_category_name     : str  — nhóm lớn, vd 'Điện tử - Điện máy', 'Thời trang & Phụ kiện'
        sl_sub_category_name : str  — danh mục con, vd 'Điện thoại & Phụ kiện', 'Quần áo nữ'
        brand                : str  — vd 'APPLE', 'SAMSUNG', 'OTHER'

    Tùy chọn:
        model                : str  — tên model / sản phẩm
        capacity             : str  — vd '128 GB' (chỉ dùng cho điện tử)
        sl_condition         : str  — 'New' | 'Like New' | 'Good'  (mặc định 'Good')
        origin_label         : str  — 'Chính hãng' | 'Chính hãng VN' | 'Nhập khẩu' | 'Không rõ'…
        warranty_label       : str  — 'Còn bảo hành' | 'Hết bảo hành' | 'Không bảo hành'
        manufacture_year     : int  — vd 2022
        region_name          : str  — tỉnh/thành, vd 'Tp Hồ Chí Minh'

    Returns
    -------
    dict: { price_vnd, price_display, confidence_range: {low, high} }
    """
    sub_cat = input_data.get('sl_sub_category_name', 'Unknown')
    is_elec = 1 if sub_cat in ELECTRONICS_CATS else 0

    cap_str = input_data.get('capacity', '') if is_elec else ''
    cap_match = pd.Series([cap_str]).str.extract(r'(\d+)')[0]
    capacity_gb = float(cap_match.iloc[0]) if pd.notna(cap_match.iloc[0]) else 0.0

    year = int(input_data.get('manufacture_year', 0))

    row = {
        'sl_category_name':     input_data.get('sl_category_name', 'Unknown'),
        'sl_sub_category_name': sub_cat,
        'brand':                input_data.get('brand', 'OTHER'),
        'model':                input_data.get('model', 'Unknown'),
        'capacity_gb':          capacity_gb,
        'is_electronics':       is_elec,
        'sl_condition':         input_data.get('sl_condition', 'Good'),
        'origin_label':         input_data.get('origin_label', 'Không rõ'),
        'warranty_label':       input_data.get('warranty_label', 'Không bảo hành'),
        'manufacture_year':     year,
        'product_age':          CURRENT_YEAR - year if year > 2000 else 5,
        'region_name':          input_data.get('region_name', 'Unknown'),
    }

    X_input = pd.DataFrame([row])

    for col in CAT_COLS:
        le = label_encoders[col]
        val = str(X_input[col].iloc[0])
        X_input[col] = le.transform([val]) if val in le.classes_ else le.transform([le.classes_[0]])

    pred_log   = model.predict(X_input[FEATURE_COLS])[0]
    pred_price = np.expm1(pred_log)
    margin     = pred_price * (mape / 100)

    return {
        'price_vnd':        int(pred_price),
        'price_display':    f"{pred_price:,.0f} đ",
        'confidence_range': {
            'low':  f"{max(0, pred_price - margin):,.0f} đ",
            'high': f"{pred_price + margin:,.0f} đ",
        },
    }


# ── Demo: đa dạng danh mục ────────────────────────────────────────────────────
examples = [
    # --- Electronics ---
    dict(sl_category_name='Điện tử - Điện máy',
         sl_sub_category_name='Điện thoại & Phụ kiện',
         brand='APPLE', model='iPhone 14 Pro Max', capacity='256 GB',
         sl_condition='Good', origin_label='Chính hãng VN',
         warranty_label='Còn bảo hành', manufacture_year=2022,
         region_name='Tp Hồ Chí Minh'),

    dict(sl_category_name='Điện tử - Điện máy',
         sl_sub_category_name='Máy tính & Laptop',
         brand='DELL', model='XPS 15', capacity='512 GB',
         sl_condition='Like New', origin_label='Nhập khẩu',
         warranty_label='Còn bảo hành', manufacture_year=2023,
         region_name='Hà Nội'),

    dict(sl_category_name='Điện tử - Điện máy',
         sl_sub_category_name='Máy ảnh & Quay phim',
         brand='SONY', model='Alpha A7 III',
         sl_condition='Good', origin_label='Chính hãng',
         warranty_label='Hết bảo hành', manufacture_year=2021,
         region_name='Tp Hồ Chí Minh'),

    # --- Fashion ---
    dict(sl_category_name='Thời trang & Phụ kiện',
         sl_sub_category_name='Giày dép',
         brand='OTHER', model='Nike Air Force 1',
         sl_condition='Like New', origin_label='Không rõ',
         warranty_label='Không bảo hành', region_name='Hà Nội'),

    dict(sl_category_name='Thời trang & Phụ kiện',
         sl_sub_category_name='Túi xách & Balo',
         brand='OTHER', model='Louis Vuitton Neverfull MM',
         sl_condition='Good', origin_label='Không rõ',
         warranty_label='Không bảo hành', region_name='Tp Hồ Chí Minh'),

    # --- Vehicle ---
    dict(sl_category_name='Xe cộ & Phương tiện',
         sl_sub_category_name='Xe máy',
         brand='OTHER', model='Honda Wave Alpha 110',
         sl_condition='Good', origin_label='Chính hãng',
         warranty_label='Hết bảo hành', manufacture_year=2020,
         region_name='Bình Dương'),

    # --- Home ---
    dict(sl_category_name='Nhà cửa & Nội thất',
         sl_sub_category_name='Nội thất phòng khách',
         brand='OTHER', model='Sofa da 3 chỗ',
         sl_condition='Good', warranty_label='Không bảo hành',
         region_name='Tp Hồ Chí Minh'),

    # --- Sports ---
    dict(sl_category_name='Thể thao & Du lịch',
         sl_sub_category_name='Dụng cụ thể thao & Gym',
         brand='OTHER', model='Xe đạp tập thể dục',
         sl_condition='Good', warranty_label='Không bảo hành',
         region_name='Hà Nội'),
]

print(f"{'Danh mục':<30} {'Brand':<12} {'Model':<28} {'Dự đoán':>15}  Khoảng")
print('-' * 110)
for ex in examples:
    result = predict_price(ex)
    label = f"[{ex['sl_sub_category_name'][:28]}]"
    print(f"{label:<30} {ex['brand']:<12} {ex.get('model',''):<28} "
          f"{result['price_display']:>15}  "
          f"{result['confidence_range']['low']} – {result['confidence_range']['high']}")


## 12. (Optional) Hyperparameter Tuning với Optuna

In [ ]:
# Uncomment để chạy tuning (~5-10 phút)

# import optuna
# optuna.logging.set_verbosity(optuna.logging.WARNING)
#
# def objective(trial):
#     params = {
#         'n_estimators':     trial.suggest_int('n_estimators', 100, 800),
#         'max_depth':        trial.suggest_int('max_depth', 3, 9),
#         'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
#         'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
#         'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
#         'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
#         'reg_alpha':        trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
#         'reg_lambda':       trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
#         'random_state': 42, 'n_jobs': -1,
#     }
#     m = xgb.XGBRegressor(**params)
#     scores = cross_val_score(m, X_train, y_train, cv=3, scoring='r2')
#     return scores.mean()
#
# study = optuna.create_study(direction='maximize')
# study.optimize(objective, n_trials=50)
# print('Best params:', study.best_params)
# print('Best R²:', study.best_value)

print('Hyperparameter tuning (Optuna) section - uncomment to run')
